# धडा 13 - कोग्नी नॉलेज ग्राफसह एजंट मेमरी  


## सेटअप

हा नोटबुक कसा बनवायचा हे दाखवतो एक बुद्धिमान **कोडिंग सहाय्यक** ज्याला सातत्याने स्मृती आहे [**Cognee**](https://www.cognee.ai/) ज्ञान ग्राफ्स आणि **मायक्रोसॉफ्ट एजंट फ्रेमवर्क** (MAF) वापरून.

Cognee अनस्ट्रक्चर्ड टेक्स्टला रूपांतरित करते संरचित, क्वेरी करण्यायोग्य ज्ञान ग्राफमध्ये ज्याला वेटर एम्बेडिंग्स बॅकअप करतात — तुमच्या एजंटला समृद्ध, संबंध-जाणारी दीर्घकालीन स्मृती देते.

### तुम्ही काय शिकाल
1. **ज्ञान ग्राफ तयार करा**: विकासक प्रोफाइल्स आणि सर्वोत्तम प्रॅक्टिसेस संरचित, क्वेरी करण्यायोग्य ज्ञानात रूपांतरित करा.
2. **Cognee चा MAF सोबत समाकलन करा**: `@tool` फंक्शन्स वापरा जेणेकरून MAF एजंट Cognee च्या ज्ञान ग्राफला क्वेरी करू शकेल.
3. **सेशन-ओळखीच्या संभाषणांचे व्यवस्थापन**: एका सेशनमध्ये अनेक प्रश्नांमध्ये संदर्भ टिकवून ठेवा.
4. **दीर्घकालीन स्मृती**: महत्त्वाचे ज्ञान सेशन्स दरम्यान जतन करा आणि नवीन संभाषणांमध्ये ते पुनः प्राप्त करा.

### पूर्व-आवश्यकता
- Python 3.9+
- स्थानिकपणे Redis चालू (`docker run -d -p 6379:6379 redis`) सेशन व्यवस्थापनासाठी
- एक LLM API की (उदा. OpenAI) — `.env` मध्ये `LLM_API_KEY` सेट करा
- `.env` मध्ये `CACHING=true` (Cognee सेशन्ससाठी आवश्यक)
- Microsoft Foundry प्रकल्प ज्यात चॅट मॉडेल डिप्लॉय केलेले आहे
- `.env` मध्ये `AZURE_AI_PROJECT_ENDPOINT` आणि `AZURE_AI_MODEL_DEPLOYMENT_NAME`
- Azure CLI प्रमाणित केलेले (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")


In [ ]:
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)

print("✅ FoundryChatClient created")


## एजंट मेमरीचे प्रकार

हा नोटबुक मुख्य धडा 13 च्या नोटबुकमधील तिघाच मेमरी प्रकार वापरतो, पण दीर्घकालीन मेमरी बॅकएंड म्हणून Cognee वापरतो:

| मेमरी प्रकार | यंत्रणा | आयुष्यकाल |
|---|---|---|
| **कार्यरत** | `agent.create_session()` (MAF) | एकच संभाषण |
| **अल्पकालीन** | Cognee सत्र कॅश (Redis) | एकच सत्र |
| **दीर्घकालीन** | Cognee ज्ञान ग्राफ + व्हेक्टर्स | कायमस्वरूपी |

### Cognee ची मेमरी स्थापत्य रचना
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## कॉनी संचय तयार करा


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## भाग 1 — ज्ञानाधार तयार करणे

आमच्या कोडिंग सहाय्यासाठी एक सर्वसमावेशक ज्ञानाधार तयार करण्यासाठी आम्ही तीन प्रकारचे डेटा घेतो:

1. **डेव्हलपर प्रोफाइल** — वैयक्तिक कौशल्य आणि तांत्रिक पार्श्वभूमी
2. **पायथन सर्वोत्तम पद्धती** — पायथनचा झेनसह व्यावहारिक मार्गदर्शक तत्त्वे
3. **इतिहासातील संभाषणे** — विकसक आणि AI सहाय्यक यांच्यातील मागील प्रश्नोत्तरे सत्रे


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## ज्ञान ग्राफचे दृश्य रूपांतरण करा

कॉगनी काढलेल्या घटकां आणि संबंधांचे एक परस्परसंवादी HTML दृश्य प्रस्तुत करू शकतो.


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## मेमिफायसह स्मृती समृद्ध करा

`memify()` ज्ञान ग्राफचे विश्लेषण करते आणि बुद्धिमान नियम तयार करते — पॅटर्न, सर्वोत्तम पद्धती आणि संकल्पनांमधील संबंध ओळखणे.


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## भाग 2 — Cognee उपकरणांसह MAF एजंट

आता आपण एक MAF एजंट तयार करतो जो `@tool` फंक्शन्सद्वारे Cognee च्या ज्ञान ग्राफवर क्वेरी करू शकतो. हे एजंटला ग्राफ-आधारित सेमॅंटिक शोधाची संपूर्ण शक्ती वापरण्यास परवानगी देते, तर सत्रांमधून संभाषणाचा संदर्भ राखतो.


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = provider.as_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")


## सत्रांसह कार्य करण्याची स्मृती

`AgentSession` (`agent.create_session()` द्वारे तयार केलेले) सत्राच्या आत कार्य करण्याची स्मृती प्रदान करते. एजंट पूर्वीच्या संदेशांकडे मागे पाहू शकतो आणि तेव्हाच Cognee च्या दीर्घकालीन ज्ञान ग्राफचा शोध घेऊ शकतो.


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## नवीन सत्र — दीर्घकालीन स्मृती कायम राहते

नवीन सत्र सुरू केल्याने कार्यरत स्मृती साफ होते, पण Cognee ज्ञान ग्राफ अजूनही उपलब्ध असतो. एजंट पूर्णपणे नवीन संभाषणात तेच दीर्घकालीन ज्ञान पुन्हा प्राप्त करू शकतो.


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## सारांश

या नोटबुकमध्ये तुम्ही एक कोडिंग सहाय्यक तयार केला आहे जो **MAF चे कार्य स्मरणशक्ती** (`agent.create_session()`) आणि **Cognee ची दीर्घकालीन ज्ञान ग्राफ** एकत्र करतो.

### तुम्ही काय शिकलात
1. **ज्ञान ग्राफ बांधणी**: Cognee अनस्ट्रक्चर्ड मजकूर घेतो आणि ग्राफ + व्हेक्टर स्मृती तयार करतो.
2. **memify सह ग्राफ समृद्धी**: तुमच्या विद्यमान ग्राफवरून निष्कर्षित तथ्ये आणि समृद्ध संबंध जोडले.
3. **MAF + Cognee एकत्रीकरण**: `@tool` फंक्शन्स MAF एजंटला Cognee च्या ग्राफला स्वाभाविकरीत्या क्वेरी करण्याची परवानगी देतात.
4. **कार्य स्मरणशक्ती + दीर्घकालीन स्मरणशक्ती**: `AgentSession` (`agent.create_session()` द्वारे) सेशन संदर्भ पुरवतो तर Cognee कायमस्वरूपी ज्ञान प्रदान करतो.
5. **NodeSets सोबत फिल्टर केलेला शोध**: ज्ञान ग्राफच्या विशिष्ट उपसमुहांवर लक्ष केंद्रित करा (उदा. फक्त तत्त्वे).

### मुख्य मुद्दे
- **Cognee** कच्चा मजकूर संरचित, संबंध-समजणारी स्मृतीमध्ये रूपांतरित करतो — एक सपाट व्हेक्टर स्टोअरपेक्षा अधिक सामर्थ्यशाली.
- **`@tool` फंक्शन्स** MAF एजंट आणि बाह्य ज्ञान प्रणालींमध्ये स्वच्छ सेतू बांधतात.
- **`AgentSession`** (`agent.create_session()` द्वारे) प्रत्येक संभाषणाचा संदर्भ दीर्घकालीन ज्ञानापासून वेगळा ठेवतो.
- एकाच ज्ञान ग्राफ अनेक सेशन्स आणि एजंटना सेवा पुरवतो.

### वास्तविक जगातील अनुप्रयोग
- **डेव्हलपर कोपायलट्स**: कोड पुनरावलोकन, घटना विश्लेषण, आर्किटेक्चर सहाय्यक
- **ग्राहक-सामना करणारे कोपायलट्स**: उत्पादन दस्तऐवज, FAQ, आणि CRM नोंदींवर सहाय्यक एजंट्स
- **आंतररिक तज्ञ कोपायलट्स**: धोरण, कायदेशीर, किंवा सुरक्षा सहाय्यक मार्गदर्शक सूचनांवर आधारित विवेचन
- **एकत्रित डेटा स्तर**: संरचित आणि असंरचित डेटा एकत्र करून एका क्वेरीयोग्य ग्राफमध्ये संयोजन

### पुढील पावले
- Cognee मध्ये कालिक जागरूकता अनुभव करा
- डोमेन-विशिष्ट ग्राफ गुणवत्ता साठी OWL ऑंटोलॉजी परिभाषित करा
- वेळेनुसार पुनर्प्राप्ती सुधारण्यासाठी वापरकर्ता अभिप्राय लूप जोडा
- एकाच Cognee स्मृती स्तर शेअर करणाऱ्या बहु-एजंट प्रणालींना स्केल करा


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
हा दस्तऐवज AI भाषांतर सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) चा वापर करून अनुवादित केला आहे. जरी आम्ही अचूकतेसाठी प्रयत्न करतो, तरी कृपया लक्षात घ्या की स्वयंचलित भाषांतरांमध्ये त्रुटी किंवा अचूकतेची कमतरता असू शकते. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्रोत मानला पाहिजे. महत्त्वाची माहिती असल्यास, व्यावसायिक मानवी भाषांतराची शिफारस केली जाते. या भाषांतराच्या वापरामुळे उद्भवणाऱ्या कोणत्याही गैरसमज किंवा चुकीच्या अर्थलावणीसाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
